# FLAG 2027 - Notebook 31: tune the winner (contrastive alignment) - temperature, weight, batch size, CE off, memory bank

NB28: InfoNCE alignment between the scored face and voice embeddings = CV 23.96 vs 25.18. It was run at ONE setting (w 0.5, tau 0.07, 16 identities per batch). Contrastive losses are known to be sensitive to the temperature and to the number of negatives, so this notebook sweeps exactly those, plus a cross-batch memory bank (more negatives for free; Wang et al. CVPR 2020 'XBM').

| Combo | Change vs K0 |
|---|---|
| `K0_nce_ref` | NB28 H2 (w 0.5, tau 0.07, 16 ids) - copied from NB28 |
| `K1-K4` | tau 0.03 / 0.05 / 0.10 / 0.20 |
| `K5-K7` | weight 0.25 / 1.0 / 2.0 |
| `K8, K9` | 32 / 64 identities per batch (more negatives) |
| `K10` | identity cross-entropy switched off (InfoNCE + OPL only) |
| `K11, K12` | memory bank of the last 8 / 32 batches as extra negatives |

Then **stage B**: the best 2 and K0 re-run on NEW folds with NEW seeds. Only a stage-B win counts.

**How to run:** ONE code cell. Paste it at the END of your NB28 notebook (after NB28 Cells 1-2 have run in this session). About 1.5-2 h. Resumable. Paste the **SUMMARY**.

In [ ]:
# NB31 - contrastive tuning (needs NB28 Cells 1-2 run in this session)
SUMMARY = []
def nce_x(ft, vt, lab, Qf, Qv, Ql, tau):
    """symmetric identity-aware InfoNCE where the candidates are the batch PLUS a memory bank of earlier (detached) embeddings"""
    Fa = Fn.normalize(ft, dim=1); Va = Fn.normalize(vt, dim=1)
    Vc = torch.cat([Va] + Qv) if Qv else Va; Fc = torch.cat([Fa] + Qf) if Qf else Fa; Lc = torch.cat([lab] + Ql) if Ql else lab
    same = lab[:, None] == Lc[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(Fa @ Vc.t() / tau) + one(Va @ Fc.t() / tau))
def train_head3(F_raw, V_raw, all_tr, vsets, seed, cfg):
    w_al, tau, pid, w_ce, mem = cfg.get('w', 0.5), cfg.get('tau', 0.07), cfg.get('pid', P_ID), cfg.get('ce', 1.0), cfg.get('mem', 0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    Qf, Qv, Ql = [], [], []
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), pid, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = w_ce * Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if mem:
                loss = loss + w_al * nce_x(ft, vt, lab, Qf, Qv, Ql, tau)
                Qf.append(Fn.normalize(ft, dim=1).detach()); Qv.append(Fn.normalize(vt, dim=1).detach()); Ql.append(lab)
                Qf, Qv, Ql = Qf[-mem:], Qv[-mem:], Ql[-mem:]
            else:
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
CFG31 = {'K0_nce_ref': {}, 'K1_tau0.03': dict(tau=0.03), 'K2_tau0.05': dict(tau=0.05), 'K3_tau0.10': dict(tau=0.10), 'K4_tau0.20': dict(tau=0.20),
         'K5_w0.25': dict(w=0.25), 'K6_w1.0': dict(w=1.0), 'K7_w2.0': dict(w=2.0), 'K8_pid32': dict(pid=32), 'K9_pid64': dict(pid=64),
         'K10_noCE': dict(ce=0.0), 'K11_mem8': dict(mem=8), 'K12_mem32': dict(mem=32)}
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV31 = os.path.join(RESDIR, 'nb31_cv.csv')
if not os.path.exists(CSV31):
    r = pd.read_csv(os.path.join(RESDIR, 'nb28_cv.csv')); r = r[r.combo == 'H2_align_nce'][['seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    r.insert(0, 'combo', 'K0_nce_ref'); r.insert(0, 'stage', 'A'); r.to_csv(CSV31, index=False); print('K0 copied from NB28 H2 (%d runs)' % len(r))
done31 = pd.read_csv(CSV31); t0 = time.time()
def run31(stage, names, seeds, fold_data):
    global done31
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done31.stage == stage) & (done31.combo == name) & (done31.seed == seed) & (done31.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]
                model, best_ep, XF, XV = train_head3(F_RAW, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed, CFG31[name])
                e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV31, mode='a', header=False, index=False); done31 = pd.concat([done31, row])
            print('%s %-12s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab31(stage):
    R = pd.read_csv(CSV31); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
run31('A', list(CFG31), [1, 2, 3], FOLD_DATA)
TA = tab31('A'); refA = TA.loc['K0_nce_ref']; TOP = [c for c in TA.index if c != 'K0_nce_ref'][:2]
run31('B', ['K0_nce_ref'] + TOP, [4, 5, 6], FOLD_B)
TB = tab31('B'); refB = TB.loc['K0_nce_ref']
note('NB31 STAGE A (usual folds, seeds 1-3; system C features + InfoNCE; C reference without InfoNCE was 25.18):'); note(TA.to_string())
note('NB31 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
for c in TOP:
    dB = TB.loc[c, 'overall'] - refB['overall']
    note('%s: stage A %+.2f | stage B %+.2f (std %+.2f, gender %+.2f) vs K0 -> %s' % (c, TA.loc[c, 'overall'] - refA['overall'], dB, TB.loc[c, 'std'] - refB['std'], TB.loc[c, 'gender'] - refB['gender'], 'ADOPT' if dB <= -0.3 else 'keep K0'))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)